# link of the dataset
https://www.kaggle.com/datasets/mahmoudreda55/satellite-image-classification

# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Path of Data

**I split the data in my PC and i used it in my jupyter, but it is the same data**

In [ ]:
train_path = r'C:\Users\Mohamed Hamde\Data Science\Projects Deep Learning/SatelliteImageSplit\train'
val_path = r'C:\Users\Mohamed Hamde\Data Science\Projects Deep Learning/SatelliteImageSplit\val'
test_path = r'C:\Users\Mohamed Hamde\Data Science\Projects Deep Learning/SatelliteImageSplit\test'

# Get the Data and Resize it 

In [ ]:
train_batches = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet.preprocess_input).flow_from_directory(
    train_path, target_size=(224, 224), batch_size=3941)

val_batches = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet.preprocess_input).flow_from_directory(
    val_path, target_size=(224, 224), batch_size=1126)

test_batches = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet.preprocess_input).flow_from_directory(
    test_path, target_size=(224, 224), batch_size=564)

# Classes of Data

In [ ]:
classes = train_batches.class_indices
classes

In [ ]:
labels = []
for key, value in classes.items():
    labels.append(key)

# Split the Data for images and labels

In [ ]:
X_train, y_train = next(train_batches)
X_val, y_val = next(val_batches)
X_test, y_test = next(test_batches)

# Show some of images

In [ ]:
plt.figure(figsize=(20,20))
for n, i in enumerate(np.random.randint(0, len(X_train), 25)):
    plt.subplot(5, 5, n+1)
    plt.imshow((X_train[i]*255).astype(np.uint8))
    plt.xlabel(labels[np.argmax(y_train[i])])

# Using ImageDataGenerator

In [ ]:
img_train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True)

train_generator = img_train_datagen.flow(X_train, y_train, batch_size=30)
valid_generator = img_train_datagen.flow(X_val, y_val)

# Calling the Model (Mobile Net)

In [ ]:
mobilnet = tf.keras.applications.mobilenet.MobileNet(dropout=0.4)

# Freezing the Layers except the output

In [ ]:
model_mobilenet = tf.keras.models.Sequential()
for layer in mobilnet.layers[:-1]:
    model_mobilenet.add(layer)

In [ ]:
for layer in model_mobilenet.layers:
    layer.trainable = False

# Create the output layer

In [ ]:
model_mobilenet.add(tf.keras.layers.Dense(4, activation='softmax'))

# Summary of Model

In [ ]:
model_mobilenet.summary()

In [ ]:
tf.keras.utils.plot_model(model_mobilenet, to_file='model.jpg')

# Optimization the Model

In [ ]:
model_mobilenet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Using the Callbacks

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(patience=10)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=0.0001)

# Fitting the Model

In [ ]:
history = model_mobilenet.fit(train_generator,  batch_size=64, epochs=4, validation_data=valid_generator ,verbose=1, callbacks=[reduce_lr, early_stopping])

# Prediction and Evaluate the Model

In [ ]:
predictions = model_mobilenet.predict(X_test, verbose=0)

In [ ]:
acc = model_mobilenet.evaluate(X_test, y_test)
print(acc)

# Plotting the Results to know if there is Overfitting or no!

In [ ]:
plt.plot(history.history['val_loss'], color='red', label='val_loss')
plt.plot(history.history['loss'], color='blue', label='train_loss')
plt.xlabel('Epochs')
plt.ylabel("Loss")
plt.legend()

In [ ]:
plt.plot(history.history['val_accuracy'], color='red', label='val_accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train_accuracy')
plt.xlabel('Epochs')
plt.ylabel("Loss")
plt.legend()

# Show the Predictions

In [ ]:
plt.figure(figsize=(20, 20))
for n, i in enumerate(np.random.randint(0, len(predictions), 20)):
    plt.subplot(5, 5, n+1)
    plt.imshow((X_test[i]*255).astype(np.uint8))
    plt.xlabel(labels[np.argmax(predictions[i])])

# Confusion Matrix

In [ ]:
conf = confusion_matrix(y_test.argmax(axis=1), predictions.argmax(axis=1))

In [ ]:
sns.heatmap(conf, annot= True, fmt='d')

# Classification Report

In [ ]:
clas = classification_report(y_test.argmax(axis=1), predictions.argmax(axis=1))
print(clas)